In [ ]:
import asyncio
import aiohttp
import csv
from bs4 import BeautifulSoup
import time
import random

# --- ⚙️ CONFIGURATION ⚙️ ---
TOURNAMENTS = [
    {"id": "4626469", "name": "PBL", "date": "09/06/2025"},
    {"id": "9460497", "name": "UBL 1-1", "date": "11/08/2025"},
    {"id": "608123", "name": "UBL 1-2", "date": "11/15/2025"},
    {"id": "8762799", "name": "UBL 1-3", "date": "11/23/2025"},
    {"id": "5669429", "name": "UBL 1-4", "date": "11/29/2025"},
    {"id": "7826390", "name": "UBL 1-5", "date": "11/30/2025"}
]

# Set a concurrency limit. 10 is a safe start to avoid getting IP banned.
MAX_CONCURRENT_REQUESTS = 10 

async def fetch_ubl_player(semaphore, player_dict, tid, name, date, base_url, profile_url):
    """Async worker to fetch a single player's data using dynamic cookie injection."""
    current_id = player_dict['id']
    dropdown_txt = player_dict['dropdown_text']

    async with semaphore:
        await asyncio.sleep(random.uniform(0.2, 0.7))
        
        # 💉 CONSTRUCT THE EXACT COOKIE FROM YOUR SCREENSHOT
        # Name: TCG + Tournament ID
        # Value: ShikibetsuNo + Player ID + &Visitor=
        cookie_string = f"TCG{tid}=ShikibetsuNo={current_id}&Visitor="
        
        headers = {
            'User-Agent': 'Mozilla/5.0', 
            'Referer': base_url,
            'Cookie': cookie_string
        }
        
        # Create session with our spoofed headers
        async with aiohttp.ClientSession(headers=headers) as session:
            try:
                # Go straight to the profile page
                async with session.get(profile_url) as profile_resp:
                    html_bytes = await profile_resp.read()
                    
                soup = BeautifulSoup(html_bytes, 'html.parser', from_encoding='utf-8')

                found_name = None
                target_str = f"'{current_id}'"
                target_row = soup.find('tr', onclick=lambda x: x and target_str in x)

                if target_row:
                    cells = target_row.find_all('td')
                    if len(cells) >= 2:
                        found_name = cells[1].get_text(strip=True)

                if found_name:
                    print(f"✅ {dropdown_txt} {found_name}")
                    return [dropdown_txt, found_name, current_id, tid, name, date]
                
                return None

            except Exception as e:
                print(f"⚠️ Error on ID {dropdown_txt}: {e}")
                return None

async def scrape_ubl_tournament(tid, name, date, csv_filename):
    print(f"📊 Scraping Tournament: {name} ({date}) with ID: {tid}")

    BASE_URL = f"https://tcg.sfc-jpn.jp/loginnum.asp?tid={tid}&MMP=&flu=&Exclusive=0"
    PROFILE_URL = f"https://tcg.sfc-jpn.jp/tour.asp?tid={tid}&kno=1&znt=1&MMP=&flu=&Exclusive=0"
    
    dropdown_results = []
    
    # Fetching the dropdown list can be done synchronously or with a quick one-off async session
    async with aiohttp.ClientSession() as session:
        try:
            async with session.get(BASE_URL) as resp:
                html_bytes = await resp.read()
                soup = BeautifulSoup(html_bytes, 'html.parser')

                select = soup.find('select', attrs={'name': 'SelectShikibetsuNo'})
                if not select:
                    print("❌ Error: Dropdown not found.")
                    return

                for opt in select.find_all('option'):
                    val = opt.get('value')
                    text = opt.get_text(strip=True)
                    if val and val.strip():
                        dropdown_results.append({'id': val, 'dropdown_text': text})
        except Exception as e:
            print(f"❌ Connection Error: {e}")
            return

    print(f"Found {len(dropdown_results)} players in dropdown. Starting async extraction...")

    # Set up the semaphore to limit concurrency
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    
    # Create a list of tasks for all players
    tasks = [
        fetch_ubl_player(semaphore, p, tid, name, date, BASE_URL, PROFILE_URL) 
        for p in dropdown_results
    ]

    # Run all tasks concurrently and wait for them to finish
    results = await asyncio.gather(*tasks)

    # Filter out any None values (failed requests or missing names)
    players_in_tournament = [res for res in results if res is not None]

    # Save to CSV
    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if csvfile.tell() == 0:
            writer.writerow(['Dropdown Text', 'Player Name', 'Player ID', 'Tournament ID', 'Tournament Name', 'Tournament Date'])
        writer.writerows(players_in_tournament)
        print(f"💾 Saved {len(players_in_tournament)} records to {csv_filename}")

async def scrape_tournament(tid, name, date, csv_filename):
    if "UBL" in name:
        await scrape_ubl_tournament(tid, name, date, csv_filename)
    elif "PBL" in name:
        print(f"⚠️ Skipping PBL tournament: {name}")
    elif "MBL" in name:
        print(f"⚠️ Skipping MBL tournament: {name}")
    else:
        print(f"⚠️ No scraper defined for tournament type: {name}")

async def run_scraper():
    print("--- 🚀 Starting Optimized Async Scraper ---")
    
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    csv_filename = f"tournament_players_{timestamp}.csv"
    
    for tournament in TOURNAMENTS:
        if tournament['name'] in ["PBL", "MBL"]:
            continue
        print(f"\n🎯 Processing Tournament: {tournament['name']} ({tournament['date']})")
        await scrape_tournament(tournament['id'], tournament['name'], tournament['date'], csv_filename)

if __name__ == "__main__":
    await run_scraper()

--- 🚀 Starting Optimized Async Scraper ---

🎯 Processing Tournament: UBL 1-1 (11/08/2025)
📊 Scraping Tournament: UBL 1-1 (11/08/2025) with ID: 9460497
Found 258 players in dropdown. Starting async extraction...
✅ ph03540153 Jann Russell Ng
✅ ph03806355 Ellan Sanchez
✅ ph03198326 Mich
✅ ph00891794 Gideon Ong
✅ ph04510854 Lorenz Matthew Tech
✅ ph00497299 Melvyn Germono
✅ ph03361636 Jeremie Pangan
✅ ph01304382 Marwin Adrias
✅ ph05914113 Raphael Yvan Tan
✅ ph08665642 Jay Phillip Bonillo
✅ ph07857878 ShawnYG
✅ ph09798881 Jhed Mark
✅ ph08659166 Angelo Asonto
✅ ph07975647 Arthur Lance Armada
✅ ph09159382 Paolo Miguel Adriano
✅ ph09824765 Kenneth Castillo
✅ ph10660363 Jhon Carlo Agpalza
✅ ph10504706 Emmanuelle Eje
✅ ph13469953 Allan Kristoffer Agtutubo Velasquez
✅ ph14487390 JJPF
✅ ph11796696 Laurence Leona
✅ ph15249465 Irwin Cristan Soriano
✅ ph14732383 Bryan Anthony Torre
✅ ph16206277 Rod
✅ ph16484152 Rafael George Filasol
✅ ph15920178 Kath Agbayani
✅ ph17873488 Yllaine Fealiz Lois Laggui Sa

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


✅ ph79063973 Oliver Ignacio
✅ ph82508568 Christian Hilario
✅ ph79694762 JV Miranda
✅ ph82631028 Ignacio Santiago III
✅ ph80208858 FJP
✅ ph80930331 Eleanore Teo
✅ ph82497217 Jose Lorenzo Baylon
✅ ph79621888 Stephen Edward Chua
✅ ph82937697 Evhan Will
✅ ph83004506 Jed Tapiador


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


✅ ph83163115 Natalie Mae Caumban
✅ ph84249065 Aaron Gabriel Valentona
✅ ph87654990 Gilbert Ong
✅ ph84628327 Ludivic
✅ ph85746467 Kristian Jesson Malong
✅ ph84129798 Reino Labian
✅ ph87746199 Juan Miguel Candelaria Roman
✅ ph84597039 Jose Mari Antonio Rabang
✅ ph88580121 Aiden Saldana
✅ ph88257795 Patrick Angelo Liwanag
✅ ph90108890 Mikko
✅ ph89364048 Christian Dorotan
✅ ph89842964 Bernard Piguing
✅ ph89888752 Gian
✅ ph91127717 Louis Tan
✅ ph90585491 marlon vicente
✅ ph90163117 Andrei Philippe Lim
✅ ph91897557 AAA Xiayi Lin
✅ ph91966165 JECA
✅ ph90637789 Darakin
✅ ph91688207 John Dexter
✅ ph92174097 florencio IV Tan
✅ ph93369457 Randell Marc Santos
✅ ph94450750 Martin Euan B Garcia
✅ ph93229303 John Vincent R Oblefias
✅ ph94991514 Gabrielle Antulav de Leon
✅ ph92323019 Laurrie Vernon Funelas
✅ ph94707437 Louie Sid Maguigad
✅ ph96104226 Garo marc john tyrell
✅ ph95485627 David Glen Bantug
✅ ph95280670 Christian Sky T Mendoza
✅ ph96088340 Sean Ragas
✅ ph97855330 Watanabe Ko
✅ ph98133359 J

...

✅ ph91897557 AAA Xiayi Lin
✅ ph92080444 Angelo


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


✅ ph95280670 Christian Sky T Mendoza
✅ ph96104226 Garo marc john tyrell
✅ ph96004726 matthew kenneth lee
✅ ph96636778 Neil Adrian Zabala
✅ ph99511813 Ron Daniel Manguerra


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


✅ ph96536247 Glenn Nathaniel Eleazar
✅ ph96160563 Lorenz rances
✅ ph99865347 Lucas
💾 Saved 129 records to tournament_players_20260405-162423.csv
⚠️ No scraper defined for tournament type: UBL 1-3

🎯 Processing Tournament: UBL 1-4 (11/29/2025)
📊 Scraping Tournament: UBL 1-4 (11/29/2025) with ID: 5669429
Found 260 players in dropdown. Starting async extraction...
✅ ph03529225 Joseph Capistrano
✅ ph00891794 Gideon Ong
✅ ph04209791 Landon David
✅ ph01304382 Marwin Adrias
✅ ph01185132 Adrian Ted Yu
✅ ph04003838 Bryce Nathan Mendoza Paradiang
✅ ph01165132 Jude Alonzo
✅ ph03860906 Jhoanna Grace Fadrigo
✅ ph00175656 Kristofer Neilsen
✅ ph03361636 Jeremie Pangan
✅ ph08659166 Angelo Asonto
✅ ph07388100 Raymundo Balido
✅ ph04510854 Lorenz Matthew Tech
✅ ph08512059 Berland  Lagutin
✅ ph04567306 Mac Bordallo
✅ ph05267410 Jay
✅ ph06421126 Rem Carlo Cabungcal
✅ ph08665642 Jay Phillip Bonillo
✅ ph11013899 Mark Neil Jesun Loreto
✅ ph09041663 Johan Daniel Koizumi
✅ ph13267785 Kirby Cunanan
✅ ph11060587 